In [ ]:
from keras.models import Sequential
import tensorflow as tf
from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Conv2DTranspose, Cropping2D, UpSampling2D

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.models import Model


In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
input_shape = (400,320, 3)

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)
    
path_train = "/path/to/private-data/Datagen_Right/Inlet (5 marking)/train"
path_val = "/path/to/private-data/Datagen_Right/Inlet (5 marking)/validation"

batch_size = 32
target_size = (input_shape[0],input_shape[1])
train_generator = datagen.flow_from_directory(path_train, target_size=target_size,  batch_size=batch_size, class_mode='input', shuffle=True, color_mode='rgb')

validation_generator = datagen.flow_from_directory(path_val, target_size=target_size , batch_size=batch_size, class_mode='input', shuffle=True, color_mode='rgb')

In [ ]:
model = Sequential()
# Encoder
model.add(Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=input_shape))
model.add(MaxPooling2D((2, 2), padding='same'))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D((2, 2), padding='same'))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D((2, 2), padding='same'))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D((2, 2), padding='same'))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D((2, 2), padding='same'))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D((2, 2), padding='same'))
model.add(Conv2D(8, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D((2, 2), padding='same'))

model.save('Latent_space_(inlet_5_right).keras')

# Decoder
model.add(Conv2DTranspose(256, (3, 3), strides=(2, 2), activation='relu', padding='same'))
model.add(Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same'))
model.add(Conv2DTranspose(128, (3, 3), strides=(2, 2), activation='relu', padding='same'))
model.add(Conv2DTranspose(32, (3, 3), strides=(2, 2), activation='relu', padding='same'))
model.add(Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same'))
model.add(Conv2DTranspose(128, (3, 3), strides=(2, 2), activation='relu', padding='same'))
model.add(Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same'))
model.add(Conv2D(3, (3, 3), activation='sigmoid', padding='same'))
model.add(Cropping2D(cropping=((56,56), (32,32))))

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
epochs = 1000
steps_per_epoch = len(train_generator)
model.fit(train_generator, epochs=epochs, validation_data=validation_generator, batch_size=batch_size, steps_per_epoch=steps_per_epoch)

In [ ]:
#model.save('full_model(inlet_5_right).keras')

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('full_model(inlet_5_right).keras')

In [ ]:
import seaborn as sns
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
import matplotlib.pyplot as plt



#image_path = "/path/to/private-data/Croped_Pictures/train/Right/Inlet (3 marking)/example.bmp" # inlet 3
image_path = "/path/to/private-data/Croped_Pictures/train/Right/Inlet (5 marking)/example.bmp"    # inlet 5
#image_path = "/path/to/private-data/Croped_Pictures/train/Right/Outlet/example.bmp"   # Outlet

img = load_img(image_path)
plt.imshow(img)
img = np.expand_dims(img,axis=0)
img = img/255.0
np.shape(img)


In [ ]:
prediction = model.predict(img)
prediction = np.squeeze(prediction)
plt.imshow(prediction)

In [ ]:
import glob, os
 
in3_mse = []
in5_mse = []
outlet_mse = []

in3_cos = []
in5_cos = []
outlet_cos = []

path = "/path/to/private-data/Croped_Pictures/train/Right/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        img = load_img(image_path)
        img = np.expand_dims(img,axis=0)
        img = img/255.0
        prediction = model.predict(img)      
        mse = np.mean((img - prediction) ** 2)
        
        vector1 = img.squeeze()
        vector2 = prediction.squeeze()
        vector1 = vector1.flatten()
        vector2 = vector2.flatten()
        vector1 = vector1 / np.linalg.norm(vector1)
        vector2 = vector2 / np.linalg.norm(vector2)
        cosine_similarity = np.dot(vector1, vector2)
        
        if label == "Inlet (3 marking)":
            in3_mse.append(mse)
            in3_cos.append(cosine_similarity)
            
        elif label == "Inlet (5 marking)":
            in5_mse.append(mse)
            in5_cos.append(cosine_similarity)
 
            
        elif label == "Outlet":
            outlet_mse.append(mse)
            outlet_cos.append(cosine_similarity)

In [ ]:
val_in3_cos = []
val_in5_cos = []
val_outlet_cos = []
val_others_cos = []
 
val_in3_mse = []
val_in5_mse = []
val_outlet_mse = []
val_others_mse =[]

path = "/path/to/private-data/Croped_Pictures/validation/Right/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        img = load_img(image_path)
        img = np.expand_dims(img,axis=0)
        img = img/255.0
        prediction = model.predict(img)      
        mse = np.mean((img - prediction) ** 2)

        vector1 = img.squeeze()
        vector2 = prediction.squeeze()
        vector1 = vector1.flatten()
        vector2 = vector2.flatten()
        vector1 = vector1 / np.linalg.norm(vector1)
        vector2 = vector2 / np.linalg.norm(vector2)
        cosine_similarity = np.dot(vector1, vector2)   
        
        if label == "Inlet (3 marking)":
            val_in3_mse.append(mse)
            val_in3_cos.append(cosine_similarity)
           
            
            
        elif label == "Inlet (5 marking)":
            val_in5_mse.append(mse)
            val_in5_cos.append(cosine_similarity)

            
        elif label == "Outlet":
            val_outlet_mse.append(mse)
            val_outlet_cos.append(cosine_similarity)
            
        elif label == "Others":
            val_others_cos.append(cosine_similarity)
            val_others_mse.append(mse)

In [ ]:
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_mse, in5_mse, val_in5_mse, outlet_mse, val_others_mse])

# Setting labels
plt.xlabel('Class')
plt.ylabel('MSE Scores')
plt.title('Box Plot of MSE Scores for Different Classes')
plt.xticks([0, 1, 2, 3], ['Inlet type-1', 'Inlet Type-2', 'Inlet Type-2 (Test)', 'Outlet', 'Unmarked'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
print(max(val_in5_mse))
print(min(in3_mse))

In [ ]:
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_cos, in5_cos, val_in5_cos, outlet_cos, val_others_cos])

# Setting labels
plt.xlabel('Class')
plt.ylabel('Cosine similarity Scores')
plt.title('Box Plot of Cosine similarity Scores for Different Classes')
plt.xticks([0, 1, 2, 3], ['Inlet type-1','Inlet Type-2', 'Inlet type-2 (Test)', 'Outlet',  'Unmarked'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
print(min(val_in5_cos))
print(max(in3_cos))

In [ ]:
val_others_mse